# Case study · Five questions about the BTI decisions

*Session 1, block 2 practice. Author: course team, licence CC-BY-4.0.*

You answer five questions about a random sample of 50,000 European Binding Tariff Information (BTI) decisions with pandas. In each decision, an EU customs authority states how a product described by a trader is classified in the customs tariff; the four-digit **heading** (for example `9503`, toys) is the label the course later learns to predict. Write each answer as **one sentence** in the Markdown cell below the code that produces it.

**Before you start**, generate the data once from the repository root (downloads about 400 MB):

```bash
uv run python case-study/prepare_data.py
```

Then start JupyterLab from the repository root, for example with `uv run jupyter lab`, and open this notebook.

The questions:

1. How many decisions are there per start year and issuing country, and which period do the start dates cover?
2. What share of decisions is written in each language?
3. Which five headings are most frequent, and what are their English names?
4. What is the median length of the description in characters for each language?
5. What share of decisions is still valid, and why does it depend so strongly on the start year?

Questions 1 and 2 are worked examples. Questions 3 to 5 have hints; a reference solution is at the end. Try first.

## Setup: find the data and load it

In [ ]:
from pathlib import Path

import pandas as pd

# The notebook may be started from the repository root or from its own folder.
# Walk up the folders until we find case-study/data/.
here = Path.cwd().resolve()
root = next(p for p in [here, *here.parents] if (p / "case-study" / "data").exists())
DATA = root / "case-study" / "data"

decisions = pd.read_parquet(DATA / "train_sample.parquet")
decisions.head(3)

In [ ]:
decisions.dtypes   # check the types first: dates are timestamps, codes such as heading are text

Most descriptions are not in English. The column `keywords` holds English keywords assigned by customs; use it to understand a decision you cannot read. (Keywords are decided together with the classification, so later sessions never use them as model input.)

In [ ]:
decisions.loc[0, ["language", "description", "keywords", "heading"]].to_dict()

## Question 1 · Size, period, decisions per year and country (worked example)

In [ ]:
n_rows, n_cols = decisions.shape
first, last = decisions["start_date"].min(), decisions["start_date"].max()
print(f"{n_rows:,} decisions, {n_cols} columns, from {first:%Y-%m-%d} to {last:%Y-%m-%d}")

year = decisions["start_date"].dt.year.rename("year")
per_year_country = pd.crosstab(year, decisions["issuing_country"])
top_countries = decisions["issuing_country"].value_counts().index[:6]
per_year_country[top_countries]

*Answer:* The sample has 50,000 decisions in 14 columns, starting between January 2017 and December 2023, between about 6,400 and 8,400 per year. Germany issues more than half of them. The United Kingdom (`GB`) issues decisions until 2020 and none afterwards: after Brexit, British decisions are no longer EU decisions.

## Question 2 · Language shares (worked example)

In [ ]:
shares = decisions["language"].value_counts(normalize=True).round(3)
print(decisions["language"].nunique(), "languages")
shares.head(8)

*Answer:* 57 % of the descriptions are German, 16 % French and about 5 % each English and Dutch; the rest is spread over many other EU languages.

> **Why it matters.** A text model for this task must cope with many languages at once. Session 13 shows that character n-grams handle this better than whole words.

## Question 3 · The most frequent headings and their names

*Hints:* `value_counts()` on `heading` sorts headings by their number of decisions. Turn the first five into a DataFrame with `.head(5).rename("n").reset_index()`. Load `nomenclature.parquet` from the same folder and join it with `merge(..., on="heading", how="left")` to get the English column `heading_description`.

In [ ]:
# your code for question 3


*Answer:* ...

## Question 4 · Description length by language

*Hints:* `decisions["description"].str.len()` gives the number of characters of each description. Group this Series by `decisions["language"]` and take the `median()`. Why the median and not the mean? Look at the maximum length.

In [ ]:
# your code for question 4


*Answer:* ...

## Question 5 · Decisions still valid

*Hints:* `status` is `"VALID"` or `"INVALID"`. The mean of the boolean Series `decisions["status"] == "VALID"` is the share of valid decisions. Group it by the start year. A BTI decision is valid for three years from its start date; the data were exported in 2026.

In [ ]:
# your code for question 5


*Answer:* ...

## Checking AI-generated code

If you used an AI coding assistant, check its code before you trust the number:

- Compute one value in a second, independent way (for example a filter and `len()` instead of `value_counts()`).
- Try an edge case: what does the code do with an empty DataFrame or a missing value?
- Record the tool, the prompt and how you checked the answer in your notes (later in `AI_USE.md` of your team repository).

In [ ]:
# Example of an independent check for question 2: count the German decisions with a mask
is_german = decisions["language"] == "de"
assert is_german.sum() == decisions["language"].value_counts()["de"]
round(is_german.mean(), 3)

---

## Reference solution

Try the questions yourself first. The cells below compute all five answers.

In [ ]:
# Question 3
names = pd.read_parquet(DATA / "nomenclature.parquet")
top = decisions["heading"].value_counts().head(5).rename("n").reset_index()
top["share"] = (top["n"] / len(decisions)).round(3)
top.merge(names[["heading", "heading_description"]], on="heading", how="left")

In [ ]:
# Question 4
lengths = decisions["description"].str.len()
print("longest description:", lengths.max(), "characters")   # a few very long texts pull the mean up
print("overall: median", lengths.median(), "mean", round(lengths.mean(), 1))
lengths.groupby(decisions["language"]).median()[shares.index[:6]]

In [ ]:
# Question 5
is_valid = decisions["status"] == "VALID"
print("share still valid:", round(is_valid.mean(), 3))
is_valid.groupby(decisions["start_date"].dt.year).mean().round(3)

*Reference answers.* (3) The most frequent headings are 3926 (other articles of plastics, 4.0 %), 9503 (toys, 2.8 %), 6307 (other made-up textile articles, 2.7 %), 2106 (food preparations n.e.c.) and 4202 (bags and cases); even the most frequent heading covers only 4 % of the decisions. (4) German descriptions are the longest (median 740 characters), French ones among the shortest (272); the overall mean (644) lies above the median (587). (5) Only 3 % of the decisions are still valid, all of them from 2023 (22 % of that year): decisions are valid for three years, so everything that started before 2023 had expired when the data were exported in 2026, and most 2023 decisions had run out as well or were invalidated early.

## Next step

In block 3 the same analysis becomes a module with a class, `DecisionTable`, in [`../workspace/`](../workspace/). Its methods answer these five questions, and tests check them on six hand-made decisions.